In [ ]:
import sys
import os
sys.path.insert(0, os.path.abspath('/home/ElasticNotebook'))
%load_ext ElasticNotebook

In [ ]:
%load_ext cudf.pandas

In [ ]:
%LoadCheckpoint /home/dias-benchmarks/notebooks/erikbruin/nlp-on-student-writing-eda/src/rewritten/o4_mini_high_small/checkpoints/post_cell_8_try_2.pickle

In [ ]:
%%cudf.pandas.profile
### cell 9 ###
# Aggregate first and last discourse_type per id in one pass
disp_per_id = train.groupby('id', sort=False)['discourse_type'].agg(['first','last'])
# Total number of unique ids (GPU-side)
n_ids = train.id.nunique()

# Count first occurrences
train_first = disp_per_id['first'].value_counts()
train_first.index.name = 'discourse_type'
train_first = train_first.reset_index(name='counts_first')
train_first['percent_first'] = (train_first['counts_first'] / n_ids).round(2)

# Count last occurrences
train_last = disp_per_id['last'].value_counts()
train_last.index.name = 'discourse_type'
train_last = train_last.reset_index(name='counts_last')
train_last['percent_last'] = (train_last['counts_last'] / n_ids).round(2)

# Merge and reorder columns
train_first_last = train_first.merge(train_last, on='discourse_type', how='left')
train_first_last = train_first_last[['discourse_type','counts_first','percent_first','counts_last','percent_last']]

train_first_last